# Factoid Test Evaluation: OpenAI Only

Evaluates one OpenAI model on the 95 BioASQ Task 13B factoid test questions. It sends each question and its provided snippets to the Responses API, then uses the project's BioASQ MRR scorer. No local adapter or CUDA GPU is required.


In [ ]:
import argparse
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / 'Task-Structured Counterfactual Preference Mining']:
        if (candidate / 'cse_dpo').is_dir() and (candidate / 'src').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not locate the project root.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.normalize_set_answers import normalize_answer_surface
from src.utility.bioasq_official import evaluate_with_bioasq_java
from cse_dpo.factoid_recovery import score_factoid_rows_with_java
from src.utility.eval_dataset import build_messages, load_eval_examples

SINGLE_ANSWER_PROMPT_PATH = PROJECT_ROOT / 'prompts/factoid_single_answer_aligned.json'
SINGLE_ANSWER_PROMPT_ID = 'factoid-single-answer-aligned-v1'
_prompt_registry = json.loads(SINGLE_ANSWER_PROMPT_PATH.read_text(encoding='utf-8'))
FACTOID_SINGLE_ANSWER_INSTRUCTION = _prompt_registry['prompts'][SINGLE_ANSWER_PROMPT_ID]['instructions']['factoid']


In [ ]:
OPENAI_MODEL = 'gpt-5-mini'  # Replace with another available OpenAI model if desired.
OPENAI_API_KEY_PATH = PROJECT_ROOT / 'open_ai_api.txt'
# GPT-5 reasoning and visible text share this budget; 64 can leave no visible answer.
MAX_OUTPUT_TOKENS = 256
REASONING_EFFORT = 'minimal'
MAX_RETRIES = 8
RETRY_BASE_SECONDS = 5
REQUEST_DELAY_SECONDS = 1.5  # Stay below common request/token rate limits.
RUN_NAME = None
RESUME_OUTPUT_DIR = None  # Set to an existing evaluation directory to resume from its partial JSONL.


def read_openai_api_key(path: Path) -> str:
    lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines()]
    key = next((line for line in lines if line and not line.startswith('#')), '')
    if key.startswith('OPENAI_API_KEY='):
        key = key.split('=', 1)[1].strip()
    return key.strip().strip(chr(34)).strip(chr(39))


api_key = read_openai_api_key(OPENAI_API_KEY_PATH)
assert api_key, f'No API key found in {OPENAI_API_KEY_PATH}'
client = OpenAI(api_key=api_key)

if RESUME_OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(RESUME_OUTPUT_DIR).expanduser().resolve()
    assert OUTPUT_DIR.exists(), f'Resume directory does not exist: {OUTPUT_DIR}'
else:
    run_suffix = RUN_NAME or datetime.now().strftime('%Y%m%d_%H%M%S')
    OUTPUT_DIR = PROJECT_ROOT / 'Artifacts/Factoid_SFT/evaluations' / f'factoid_test_openai_{OPENAI_MODEL}_{run_suffix}'
    if OUTPUT_DIR.exists():
        raise FileExistsError(f'Refusing to overwrite: {OUTPUT_DIR}')
    OUTPUT_DIR.mkdir(parents=True)
PARTIAL_METRICS_PATH = OUTPUT_DIR / 'partial_per_question_metrics.jsonl'
print('OpenAI model:', OPENAI_MODEL)
print('Output directory:', OUTPUT_DIR)


In [ ]:
TEST_FILES = sorted((PROJECT_ROOT / 'data/Task13BTest').glob('*_golden.json'))
assert len(TEST_FILES) == 4, f'Expected four golden test batches, found {TEST_FILES}'

source_args = type('SourceArgs', (), {
    'question_types': ['factoid'],
    'max_summary_answers': 5,
    'max_factoid_answers': 5,
    'max_list_items': 100,
    'max_resources': 0,
    'max_resource_chars': 0,
    'resource_selection': 'first',
    'resource_granularity': 'document',
    'resource_window_mode': 'single',
    'resource_reranker_model': None,
    'resource_reranker_article_model': None,
    'resource_reranker_device': None,
    'resource_reranker_batch_size': 1,
    'local_files_only': True,
    'limit': None,
})()
test_examples = load_eval_examples(TEST_FILES, source_args, {'factoid': FACTOID_SINGLE_ANSWER_INSTRUCTION})
assert len(test_examples) == 95, f'Expected 95 factoid test questions, found {len(test_examples)}'


def flatten_aliases(value):
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        return [alias for item in value for alias in flatten_aliases(item)]
    return []


def evidence_supported(question):
    evidence = normalize_answer_surface(' '.join(
        snippet.get('text', '') for snippet in question.get('snippets', []) if isinstance(snippet, dict)
    ))
    return any(
        alias and f' {alias} ' in f' {evidence} '
        for item in flatten_aliases(question.get('exact_answer', []))
        if (alias := normalize_answer_surface(item))
    )


batch_by_id, supported_ids = {}, set()
for path in TEST_FILES:
    for question in json.loads(path.read_text(encoding='utf-8'))['questions']:
        if question.get('type') == 'factoid':
            batch_by_id[question['id']] = path.stem
            if evidence_supported(question):
                supported_ids.add(question['id'])
print(f'Test factoids: {len(test_examples)}; evidence-supported: {len(supported_ids)}')


In [ ]:
def load_jsonl(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


metrics = load_jsonl(PARTIAL_METRICS_PATH)
completed_ids = {row['question_id'] for row in metrics}
print(f'Resuming with {len(completed_ids)} completed questions.')

for example in tqdm(test_examples, desc=f'OpenAI test generation: {OPENAI_MODEL}'):
    if example.question_id in completed_ids:
        continue

    messages = build_messages(example)
    instructions = messages[0]['content']
    user_input = messages[-1]['content']

    for attempt in range(MAX_RETRIES):
        try:
            # Do not pass temperature: gpt-5-mini rejects temperature control.
            response = client.responses.create(
                model=OPENAI_MODEL,
                instructions=instructions,
                input=user_input,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                reasoning={'effort': REASONING_EFFORT},
                store=False,
            )
            prediction = (response.output_text or '').strip()
            if not prediction:
                raise RuntimeError(
                    f"OpenAI returned no visible text for {example.question_id}; "
                    f"status={getattr(response, 'status', None)}, "
                    f"incomplete_details={getattr(response, 'incomplete_details', None)}, "
                    f"usage={getattr(response, 'usage', None)}"
                )
            break
        except Exception as exc:
            if attempt + 1 == MAX_RETRIES:
                raise RuntimeError(f'OpenAI generation failed for {example.question_id}: {exc}') from exc
            wait_seconds = RETRY_BASE_SECONDS * (2 ** attempt)
            print(f'OpenAI request failed for {example.question_id} ({type(exc).__name__}); retrying in {wait_seconds}s.')
            time.sleep(wait_seconds)

    row = {
        'question_id': example.question_id,
        'question_type': 'factoid',
        'body': example.body,
        'source_path': example.source_path,
        'batch': batch_by_id[example.question_id],
        'is_evidence_supported': example.question_id in supported_ids,
        'prediction': prediction,
        'response_status': getattr(response, 'status', None),
        'response_id': getattr(response, 'id', None),
        'gold_output': example.gold_output,
    }
    metrics.append(row)
    completed_ids.add(example.question_id)
    with PARTIAL_METRICS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row) + '\n')
    time.sleep(REQUEST_DELAY_SECONDS)

example_by_id = {example.question_id: example for example in test_examples}
for row in metrics:
    example = example_by_id[row['question_id']]
    row.update(question_type='factoid', body=example.body, source_path=example.source_path)
official_payload = evaluate_with_bioasq_java(
    prediction_rows=metrics,
    examples_by_key={(qid, 'factoid'): example for qid, example in example_by_id.items()},
    model_label=f'openai_{OPENAI_MODEL}', model_dir=OUTPUT_DIR,
    args=argparse.Namespace(
        bioasq_java_jar=str(PROJECT_ROOT / 'third_party/Evaluation-Measures/flat/BioASQEvaluation/dist/BioASQEvaluation.jar'),
        bioasq_java_heap='512m', bioasq_java_version=5),
    include_per_question=True,
)
official_by_id = {row['question_id']: row for row in official_payload['per_question']}
official_factoid_metrics = official_payload['aggregate']['by_type']['factoid']['metrics']
for row in metrics:
    score = official_by_id[row['question_id']]
    row.update(mrr=score['mrr'], strict_accuracy=score['strict_accuracy'],
               lenient_accuracy=score['lenient_accuracy'], scoring_backend='bioasq_java')

metrics_df = pd.DataFrame(metrics)
supported_df = metrics_df[metrics_df['is_evidence_supported']]
unsupported_df = metrics_df[~metrics_df['is_evidence_supported']]
supported_rows = supported_df.to_dict(orient='records')
unsupported_rows = unsupported_df.to_dict(orient='records')
supported_official = score_factoid_rows_with_java(
    supported_rows, example_by_id, OUTPUT_DIR/'supported_subset_official', PROJECT_ROOT,
    model_label=f'openai_{OPENAI_MODEL}-supported-subset') if supported_rows else None
unsupported_official = score_factoid_rows_with_java(
    unsupported_rows, example_by_id, OUTPUT_DIR/'unsupported_subset_official', PROJECT_ROOT,
    model_label=f'openai_{OPENAI_MODEL}-unsupported-subset') if unsupported_rows else None
summary = {
    'model_name': f'openai_{OPENAI_MODEL}',
    'scoring_backend': 'bioasq_java',
    'test_question_count': len(metrics_df),
    'test_mrr': official_factoid_metrics['mrr'],
    'test_strict_accuracy': official_factoid_metrics['strict_accuracy'],
    'test_lenient_accuracy': official_factoid_metrics['lenient_accuracy'],
    'evidence_supported_count': len(supported_df),
    'evidence_supported_mrr': supported_official['mrr'] if supported_official else None,
    'unsupported_count': len(unsupported_df),
    'unsupported_mrr': unsupported_official['mrr'] if unsupported_official else None,
}
metrics_df.to_csv(OUTPUT_DIR / 'per_question_metrics.csv', index=False)
metrics_df.to_json(OUTPUT_DIR / 'per_question_metrics.json', orient='records', indent=2)
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps({
    'model': OPENAI_MODEL,
    'test_files': [str(path) for path in TEST_FILES],
    'max_output_tokens': MAX_OUTPUT_TOKENS,
    'reasoning_effort': REASONING_EFFORT,
    'prompt_id': SINGLE_ANSWER_PROMPT_ID,
    'store': False,
    'scoring_backend': 'bioasq_java',
}, indent=2), encoding='utf-8')
pd.DataFrame([summary])


In [ ]:
metrics_df.groupby('batch', as_index=False).agg(
    question_count=('question_id', 'count'),
    mrr=('mrr', 'mean'),
    strict_accuracy=('strict_accuracy', 'mean'),
)
